# 03. Что приходит в трекер в реальном времени

Если партитура — **статика** (известна заранее, `score.json`), то исполнение — **поток событий**. На каждое нажатие клавиши приходит MIDI-сообщение `note_on` с двумя полями:

- `pitch` — номер MIDI-ноты, 0..127,
- `timestamp` — момент времени (для записанного `.mid` — секунды от начала файла; для живой игры — `time.time()`).

Это и есть **единственный канал**, по которому трекер видит исполнителя. Никаких велосити, никакой длительности (`note_off` сейчас игнорируется), никакого pitch-вектора — всё сводится к одному pitch на каждое событие.

В этом ноутбуке:

1. Грузим три синтетических исполнения (`ideal`, `rubato`, `noisy`) через `dataset_viewer.load_performance`.
2. Сравниваем `timestamp` с `nominal_onset` из партитуры.
3. Визуализируем rubato (отклонения от номинала) и noisy (опечатки).

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from dataset_viewer import load_performance

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True

DATA = PROJECT_ROOT / "generated_dataset"

## 1. Идеальный случай: `ideal.mid` совпадает с `ideal.json`

В идеале каждое событие исполнения соответствует ровно одному state партитуры и приходит в номинальный момент.

In [ ]:
ideal_score = json.loads((DATA / "ideal.json").read_text(encoding="utf-8"))
ideal_perf = load_performance(DATA / "ideal.mid")

rows = []
for s, p in zip(ideal_score["notes"], ideal_perf):
    rows.append({
        "score_idx": s["index"],
        "score_pitch": s["pitch"],
        "score_onset": round(sum(n["nominal_duration"] for n in ideal_score["notes"][: s["index"]]), 3),
        "perf_idx": p["index"],
        "perf_pitch": p["pitch"],
        "perf_time": p["timestamp"],
    })
pd.DataFrame(rows)

## 2. Rubato: тот же набор pitch, но разные интервалы

In [ ]:
rubato_score = json.loads((DATA / "rubato.json").read_text(encoding="utf-8"))
rubato_perf = load_performance(DATA / "rubato.mid")

score_onsets = np.cumsum([0.0] + [n["nominal_duration"] for n in rubato_score["notes"][:-1]])
perf_times = np.array([e["timestamp"] for e in rubato_perf])

fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].plot(score_onsets, label="score (nominal_onset)", marker="o")
axes[0].plot(perf_times, label="performance (timestamp)", marker="s")
axes[0].set_xlabel("note index")
axes[0].set_ylabel("time, s")
axes[0].set_title("Score vs performance time")
axes[0].legend()

deltas = perf_times - score_onsets
axes[1].bar(range(len(deltas)), deltas)
axes[1].axhline(0, color="black", linewidth=0.5)
axes[1].set_xlabel("note index")
axes[1].set_ylabel("perf_time − score_onset, s")
axes[1].set_title("Rubato deviations")
plt.tight_layout()
plt.show()

Видно, что солист сначала **ускоряется** (середина гаммы — отрицательная разница), потом **замедляется** (последние ноты с положительной разницей). Это типичный rubato-паттерн, который должен компенсировать `TempoTracker` (см. ноутбук 08), а в перспективе — предсказывать RL-агент.

## 3. Шум: лишние ноты, замены, пропуски

In [ ]:
noisy_score = json.loads((DATA / "noisy.json").read_text(encoding="utf-8"))
noisy_perf = load_performance(DATA / "noisy.mid")

print(f"score states  : {len(noisy_score['notes'])}")
print(f"perf events   : {len(noisy_perf)}")
print()
print("score pitches :", [n["pitch"] for n in noisy_score["notes"]])
print("perf pitches  :", [e["pitch"] for e in noisy_perf])

В `noisy.mid` пианист (синтетически) делает три типичные ошибки: лишние ноты (insertion), сдвиги на полутон (substitution) и пропуски (deletion). Это **главный challenge для трекера**: HMM с простым forward-обновлением может «застрять» или перепрыгнуть.

В ноутбуках 04–07 мы посмотрим как разные трекеры справляются с такими сценариями.

## 4. Связь с realtime-стеком

В живом режиме поток событий приходит через `live_midi_receiver.LiveMidiReceiver` (или `MidiEmulator` для replay-теста). API идентично тому, что выдаёт `load_performance`: словарь `{pitch, timestamp}`.

Это значит: **любой трекер, который работает на `load_performance(...)`, автоматически работает и в live**. В ноутбуках мы пользуемся `load_performance` для воспроизводимости.